# Dependency Networks and Cascade Propagation Lab


## Purpose

This lab simulates how local failures propagate through an AI-system dependency network.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 18
A = rng.uniform(0, 0.35, size=(n, n))
np.fill_diagonal(A, 0)

mask = rng.binomial(1, 0.25, size=(n, n))
A = A * mask

thresholds = rng.uniform(0.25, 0.75, size=n)

def simulate_cascade(initial_failures, buffer_strength=0.0, max_steps=10):
    state = np.ones(n, dtype=int)
    state[initial_failures] = 0

    rows = []

    for step in range(max_steps + 1):
        failed = 1 - state
        pressure = A @ failed

        rows.append({
            "step": step,
            "failed_count": int(np.sum(failed)),
            "failed_fraction": float(np.mean(failed)),
            "max_pressure": float(np.max(pressure)),
            "mean_pressure": float(np.mean(pressure)),
        })

        new_failures = (pressure > thresholds + buffer_strength) & (state == 1)

        if not np.any(new_failures):
            break

        state[new_failures] = 0

    return pd.DataFrame(rows)

baseline = simulate_cascade([0, 3], buffer_strength=0.0)
buffered = simulate_cascade([0, 3], buffer_strength=0.20)

baseline["scenario"] = "baseline"
buffered["scenario"] = "buffered"

pd.concat([baseline, buffered], ignore_index=True)

## Interpretation

Buffers and thresholds can determine whether a local failure remains contained or becomes a cascade.